In [4]:
print("Let's build GPT: from scratch, in code, spelled out.")

Let's build GPT: from scratch, in code, spelled out.


In [5]:
import torch

print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

True
True


In [6]:
#loading data
with open('A Clash of Kings.txt', 'r', encoding = 'utf-8') as f:
    text1 = f.read()

with open('a game of thrones.txt', 'r', encoding = 'utf-8') as g:
    text2 = g.read()

text = text1 + text2

In [7]:
print(text[:1000])

George R. R. Martin

A Clash of Kings

PROLOGUE
The comet’s tail spread across the dawn, a red slash that bled above the crags of
Dragonstone like a wound in the pink and purple sky.
The maester stood on the windswept balcony outside his chambers. It was here the ravens
came, after long flight. Their droppings speckled the gargoyles that rose twelve feet tall on
either side of him, a hellhound and a wyvern, two of the thousand that brooded over the walls
of the ancient fortress. When first he came to Dragonstone, the army of stone grotesques had
made him uneasy, but as the years passed he had grown used to them. Now he thought of
them as old friends. The three of them watched the sky together with foreboding.
The maester did not believe in omens. And yet . . . old as he was, Cressen had never seen
a comet half so bright, nor yet that color, that terrible color, the color of blood and flame and
sunsets. He wondered if his gargoyles had ever seen its like. They had been here so much
lon

In [8]:
print(f"The length of the characters is: {len(text)}")

The length of the characters is: 3378108


In [9]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !&(),-./0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyz{|}—‘’“”●
87


In [10]:
#creating a maping from characters to integers (character level tokenization)
stoi = {ch:i for i,ch in enumerate(chars)} #string to integers
itos = {i:ch for i,ch in enumerate(chars)}
print(stoi, "\n", itos)
encode = lambda string: [stoi[s] for s in string]
decode = lambda integer:''.join([itos[i] for i in integer])
print(encode("please release the winds of winter"))
print(decode(encode("please release the winds of winter")))

{'\n': 0, '\x0c': 1, ' ': 2, '!': 3, '&': 4, '(': 5, ')': 6, ',': 7, '-': 8, '.': 9, '/': 10, '0': 11, '1': 12, '2': 13, '3': 14, '4': 15, '5': 16, '6': 17, '7': 18, '8': 19, '9': 20, ':': 21, ';': 22, '?': 23, 'A': 24, 'B': 25, 'C': 26, 'D': 27, 'E': 28, 'F': 29, 'G': 30, 'H': 31, 'I': 32, 'J': 33, 'K': 34, 'L': 35, 'M': 36, 'N': 37, 'O': 38, 'P': 39, 'Q': 40, 'R': 41, 'S': 42, 'T': 43, 'U': 44, 'V': 45, 'W': 46, 'X': 47, 'Y': 48, 'Z': 49, '[': 50, ']': 51, 'a': 52, 'b': 53, 'c': 54, 'd': 55, 'e': 56, 'f': 57, 'g': 58, 'h': 59, 'i': 60, 'j': 61, 'k': 62, 'l': 63, 'm': 64, 'n': 65, 'o': 66, 'p': 67, 'q': 68, 'r': 69, 's': 70, 't': 71, 'u': 72, 'v': 73, 'w': 74, 'x': 75, 'y': 76, 'z': 77, '{': 78, '|': 79, '}': 80, '—': 81, '‘': 82, '’': 83, '“': 84, '”': 85, '●': 86} 
 {0: '\n', 1: '\x0c', 2: ' ', 3: '!', 4: '&', 5: '(', 6: ')', 7: ',', 8: '-', 9: '.', 10: '/', 11: '0', 12: '1', 13: '2', 14: '3', 15: '4', 16: '5', 17: '6', 18: '7', 19: '8', 20: '9', 21: ':', 22: ';', 23: '?', 24: 'A', 

In [11]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([3378108]) torch.int64
tensor([ 1, 30, 56, 66, 69, 58, 56,  2, 41,  9,  2, 41,  9,  2, 36, 52, 69, 71,
        60, 65,  0,  0, 24,  2, 26, 63, 52, 70, 59,  2, 66, 57,  2, 34, 60, 65,
        58, 70,  0,  0, 39, 41, 38, 35, 38, 30, 44, 28,  0, 43, 59, 56,  2, 54,
        66, 64, 56, 71, 83, 70,  2, 71, 52, 60, 63,  2, 70, 67, 69, 56, 52, 55,
         2, 52, 54, 69, 66, 70, 70,  2, 71, 59, 56,  2, 55, 52, 74, 65,  7,  2,
        52,  2, 69, 56, 55,  2, 70, 63, 52, 70, 59,  2, 71, 59, 52, 71,  2, 53,
        63, 56, 55,  2, 52, 53, 66, 73, 56,  2, 71, 59, 56,  2, 54, 69, 52, 58,
        70,  2, 66, 57,  0, 27, 69, 52, 58, 66, 65, 70, 71, 66, 65, 56,  2, 63,
        60, 62, 56,  2, 52,  2, 74, 66, 72, 65, 55,  2, 60, 65,  2, 71, 59, 56,
         2, 67, 60, 65, 62,  2, 52, 65, 55,  2, 67, 72, 69, 67, 63, 56,  2, 70,
        62, 76,  9,  0, 43, 59, 56,  2, 64, 52, 56, 70, 71, 56, 69,  2, 70, 71,
        66, 66, 55,  2, 66, 65,  2, 71, 59, 56,  2, 74, 60, 65, 55, 70, 74, 56,
      

In [12]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [13]:
print(len(train_data), len(val_data))

3040297 337811


In [14]:
#when we actually train the transformer on a lot of these datasets we only work with the chunks of the dataset called context length or block size

In [15]:
block_size = 8 
train_data[:block_size+1]
# so this is the first 9 characters in the sequence
#the above sequence of 9 characters this has multiple examples packed into it for example given 1 the output should be 30 and given 1 and 30 the output should be 56 and so on
#all of the characters follow each other, in the chunk of 9 characters there are 8 examples packed into it 


tensor([ 1, 30, 56, 66, 69, 58, 56,  2, 41])

In [16]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range (len(x)): 
    context = x[:t+1]
    target = y[t]
    print(f"the context is {context}, the target is {target}")
    # here we can see the context and the expected output value
    

the context is tensor([1]), the target is 30
the context is tensor([ 1, 30]), the target is 56
the context is tensor([ 1, 30, 56]), the target is 66
the context is tensor([ 1, 30, 56, 66]), the target is 69
the context is tensor([ 1, 30, 56, 66, 69]), the target is 58
the context is tensor([ 1, 30, 56, 66, 69, 58]), the target is 56
the context is tensor([ 1, 30, 56, 66, 69, 58, 56]), the target is 2
the context is tensor([ 1, 30, 56, 66, 69, 58, 56,  2]), the target is 41


In [17]:
#so we have loked at the time dimension of the input tensor now lets see the batch dimension
#we make the batch dimension because its faster than training on one single example at a time we have gpus that can fasten our computation significantly
torch.manual_seed(6967)
batch_size = 4
block_size = 8

def get_batch(split):
    #generate a small batch of data of input x and target y 
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data)- block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix ])
    y = torch.stack([data[i+1: block_size+i+1] for i in ix])
    return x,y

xb,yb = get_batch('train')
print('input:')
print(xb.shape)
print(xb)
print('targets')
print(yb.shape)

print("------------")

for b in range (batch_size):
    for t in range(block_size):
        context = xb[b, : t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()}, target is: {target}")

input:
torch.Size([4, 8])
tensor([[71,  2, 59, 56,  2, 74, 66, 72],
        [ 0, 84, 37, 66, 71,  2, 52,  2],
        [ 0, 52, 65, 55,  2, 42, 56, 69],
        [56, 70,  9,  2, 42, 52, 64,  2]])
targets
torch.Size([4, 8])
------------
when input is [71], target is: 2
when input is [71, 2], target is: 59
when input is [71, 2, 59], target is: 56
when input is [71, 2, 59, 56], target is: 2
when input is [71, 2, 59, 56, 2], target is: 74
when input is [71, 2, 59, 56, 2, 74], target is: 66
when input is [71, 2, 59, 56, 2, 74, 66], target is: 72
when input is [71, 2, 59, 56, 2, 74, 66, 72], target is: 63
when input is [0], target is: 84
when input is [0, 84], target is: 37
when input is [0, 84, 37], target is: 66
when input is [0, 84, 37, 66], target is: 71
when input is [0, 84, 37, 66, 71], target is: 2
when input is [0, 84, 37, 66, 71, 2], target is: 52
when input is [0, 84, 37, 66, 71, 2, 52], target is: 2
when input is [0, 84, 37, 66, 71, 2, 52, 2], target is: 68
when input is [0], targe

In [18]:
print(xb) #the input to the transformer

tensor([[71,  2, 59, 56,  2, 74, 66, 72],
        [ 0, 84, 37, 66, 71,  2, 52,  2],
        [ 0, 52, 65, 55,  2, 42, 56, 69],
        [56, 70,  9,  2, 42, 52, 64,  2]])


In [22]:
#creating a simple bigram language model for demonstration
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(6967)
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets = None):
        #idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) #(b,t,c) batch by time by channel
        # there is an issue here as per the pytorch documentation what pytorch expects is B,C,T ans what we have is (B,T,C)
        # so will have to reshape our logits
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            #now the spaces match as pytorch expects it 
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        #idx is (B,T)aray of indicies in the curernt context
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            #here we are calling self.idx which will endup callig self.forward function but we forward expects a targets input for loss
            #and while generating output we dont need loss and we dont have the targets either so we will make the targets an optional input for the forward funtion
            #focus only on the last time step
            logits = logits[:, -1,:] #becomes B,C
            #apply softmax to get probabbilities
            probs = F.softmax(logits, dim = 1) #(B,C)
            idx_next = torch.multinomial(probs, num_samples=1) #(B,1)
            #append sampled index to the running sequence
            idx = torch.cat((idx,idx_next), dim = 1)#(B,T+1)
        return idx 


m = BigramLanguageModel(vocab_size)
logits, loss = m(xb,yb)
print(logits.shape)
print(loss)
idx = torch.zeros((1,1),dtype = torch.long)
print(idx.item())
d = (decode(m.generate(idx, max_new_tokens=100)[0].tolist()))
print(d)


torch.Size([32, 87])
tensor(4.7666, grad_fn=<NllLossBackward0>)
0

—sm[c.&?F“aZAi8BtxS7}X1:’’”V
7Uns3c2fEuEBD.N.LA:]7Jw(Nr’vi4(E’J4)—?;heN(”M?eOcIK6Xe9n679WVmg‘:|WR 


In [24]:
#training the model
#create a ptorch optimizer
optimizer =torch.optim.AdamW(m.parameters(), lr = 1e-3) 

In [33]:
batch_size = 32
for steps in range(100000):
    #sample a batch of data
    xb, yb = get_batch("train")

    #evaluate the loss
    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    print(loss.item())    

2.4463438987731934
2.403681516647339
2.521312952041626
2.348215103149414
2.435037851333618
2.4478282928466797
2.590306282043457
2.435598373413086
2.274921178817749
2.4267783164978027
2.3791773319244385
2.463648796081543
2.455842971801758
2.4090235233306885
2.5630955696105957
2.3690571784973145
2.3797762393951416
2.355750322341919
2.47655987739563
2.5601909160614014
2.457071304321289
2.2804360389709473
2.483556032180786
2.522007703781128
2.4556756019592285
2.4500575065612793
2.379746913909912
2.336900472640991
2.410881519317627
2.4697751998901367
2.4989047050476074
2.3966729640960693
2.4180846214294434
2.5071918964385986
2.5286848545074463
2.603917360305786
2.667175531387329
2.3801369667053223
2.4250471591949463
2.3929905891418457
2.320383310317993
2.4844870567321777
2.487654209136963
2.365102529525757
2.3744208812713623
2.623753070831299
2.4573564529418945
2.4261832237243652
2.450775623321533
2.430422782897949
2.5693345069885254
2.4282641410827637
2.3924944400787354
2.3798840045928955


In [42]:
d = (decode(m.generate(idx, max_new_tokens=1000)[0].tolist()))
print(d)


r Shigeer’s ore londikelled y m, hatopeledioung helahethimam hasce. ain Jowon, htoma y. mearesamant. t l t cone. ieef It sto, sutewird aidilithens winerie f sache He fagand nd hegee t bert Ar t.
als le hofqusiow yng’say. the. votha laies. ththanbes tthino. stend hecuines
“I tast wht e thaghombrtr, wanshe Aflis
st the ld
s s s,
d d tof
“I BBeapiss, wn onthind mouscke oor sisalad Sk deaceng, ou ther. ad. y ouprelor ckid, Ars. b thans jede ase at twa cke lesoruery ingan t Thed won crn itomyomowoverish
Helf re waedof, d boubuitaidenghand Ro tong ss ar wethad, “Yepobad br aristhishel, anghelitha ederopselkint t sn}, g thesot . ginnghed Ye helafflispoolire hanir d th baind. hoougood, wiven auldulyo mis lden n thepe.”
aizotnd
Gowe pld tofre “Whenom Usle coun blimarsthe Seapisted. ard hing moushitheld atoury ais f AYon This.”
ct had oulin gersst’vewotete lke f Theche the bus hindshe

Gowathatothe d. cid bad th y Hoele
he hivove arencrowind p sertharsqut Imerirawanghe herstor st Roor.” uce sed